<font size="6">
    Script de pré-processamento de dados experimentais disponibilizados em Aimiyenkagbon (2024)
</font>

<br><br>

<font size="3">
    Este código executa as seguintes ações:
    <br><br>
    1) Lê todos os arquivos da pasta de um experimento definido;<br>
    2) Cria um Pandas DataFrame unindo os dados de forma estruturada;<br>
    3) Exporta para um arquivo com extensão definida pelo usuário;<br>
    4) Gera gráficos para análise dos dados processados.
</font>

<br><br>

<font size="4">
    Nomenclatura dos arquivos exportados em <code>\processed_data</code>:
</font>

<br>

<font size="3">
    dataset_processado_[EXPERIMENTO]_ds[FATOR_DOWNSAMPLING].[ext]
</font>

In [ ]:
import pandas as pd
import numpy as np
import scipy.io
from pathlib import Path


# --- Parâmetros de pré processamento ---
exp = 'B11' # Nome do experimento a ser processado (ex: 'B01', 'B02', 'B03', 'B04')
fator_ds = 12800 # Fator de downsampling (intervalo entre linhas mantidas do dataset original)
file_format = '.csv' # Exportar arquivo para .csv, .mat ou .parquet

# --- Diretórios base e Caminhos ---

base_original = Path('raw_data') / exp
base_processed = Path('processed_data')

vib_data_dir = base_original / 'vibrationData' # Diretório contendo os arquivos .mat de vibração
oc_data = base_original / f'{exp}_operatingConditions.csv' # Arquivo de condições operacionais do rolamento
temp_data = base_original / f'{exp}_meanTemperatures.csv' # Arquivo de temperaturas do rolamento
file_format = file_format.strip().lower().replace(".", "")
output_data = base_processed / f'dataset_processado_{exp}_ds{fator_ds}.{file_format}' # Nome do arquivo de saída
base_processed.mkdir(parents=True, exist_ok=True) # Criar diretório de saída se não existir

# Ler dados de OC e de temperaturas e compilar em dataframe de metadata
df_oc = pd.read_csv(oc_data)
df_oc['Time'] = pd.to_datetime(df_oc['Time'])

df_temp = pd.read_csv(temp_data)
df_temp['Time'] = pd.to_datetime(df_temp['Time'])

df_meta = pd.merge_asof(
    df_oc.sort_values('Time'), 
    df_temp.sort_values('Time'), 
    on='Time', 
    direction='nearest'
)

df_meta['metaTime'] = (
    df_meta['Time'] - df_meta['Time'].min()
).dt.total_seconds()

In [ ]:
# Lista e ordena os arquivos .mat para garantir que sigam a cronologia
matlab_files = sorted([f for f in vib_data_dir.iterdir() if f.suffix == '.mat'])

lista_dfs = []

# Iteração para cada arquivo .mat na pasta simultaneamente sobre os arquivos e o índice correspondente
for i, file_path in enumerate(matlab_files):
    # Extrai o identificador (ex: M0001)
    medicao_id = file_path.stem.split('_')[-1]
    
    # Carrega os dados do .mat
    mat_data = scipy.io.loadmat(file_path)
    
    # Extrai as vibrações das chaves corretas do array e aplica o downsampling
    # O flatten() transforma o array 2D/3D numa linha simples (1D)
    acc_a_g = 10 * mat_data['accHorizRear_A'].flatten()[::fator_ds]
    acc_c_g = 10 * mat_data['accHorizFrontal_C'].flatten()[::fator_ds]
    meas_time = mat_data['measTime'].flatten()[::fator_ds]
    
    # Cria o DataFrame para esta medição específica com um ponto por linha
    df_chunk = pd.DataFrame({
        'medicao_id': medicao_id,
        'Accel. Horiz. Front A / g': acc_a_g,
        'Accel. Horiz. Front C / g': acc_c_g,
        'meas_elapsed_time_s': meas_time 
    })
    
    # Assumindo que os arquivos .mat estão na mesma ordem temporal que o df_meta
    # Replica os valores das colunas do df_meta correspondente para todas as linhas do chunk
    if i < len(df_meta):
        meta_row = df_meta.iloc[i]
        
        # Para cada coluna no df_meta (Time, Loads, Speeds, Temps...), cria a coluna no df_chunk repetindo o mesmo valor escalar para as N medições de vibração
        for col in df_meta.columns:
            df_chunk[col] = meta_row[col]
            
    lista_dfs.append(df_chunk)

In [ ]:
# --- Unindo DataFrames e processamentos finais ---
df_final = pd.concat(lista_dfs, ignore_index=True)

df_final['meas_elapsed_time_s'] = df_final['meas_elapsed_time_s'].astype('float64')

# Calculando o tempo até a falha (RUL) em segundos

df_final['Time'] = df_final['metaTime'] + df_final['meas_elapsed_time_s']

df_final['time_to_failure_s'] = df_final['Time'].max() - df_final['Time']

# Drop de colunas não necessárias para a análise final (ex: colunas intermediárias de temperatura, etc)
df_final = df_final.drop(
    columns=[
        'medicao_id',
        'meas_elapsed_time_s',
        'metaTime'        
    ]).rename(columns={
        'time_to_failure_s': 'Rul_s',
        'setDynLoad / N': 'setDynLoad_N',
        'peak_dynLoad / N': 'peakDynLoad_N',
        'setStatLoad / N': 'setStatLoad_N',
        'meanAbs_statLoad / N': 'StatLoad_N',
        'setSpeed / rpm': 'setSpeed_rpm',
        'meanAbs_speed / rpm': 'Speed_rpm',
        'Mean Abs. Temp. T1 / °C': 'TempT1_C',
        'Mean Abs. Temp. T2 / °C': 'TempT2_C',
        'Mean Room Temp. / °C': 'RoomTemp_C',
        'Accel. Horiz. Front A / g': 'AccelA_g',
        'Accel. Horiz. Front C / g': 'AccelC_g'
    })

# Reordenando colunas para a visualização ideal (Macro -> Micro -> Inputs -> Sinais)
col_order = [
    # 1. Tempo Macro e Identificadores do Experimento
    'Rul_s',
    
    # 3. Condições de Operação (Variáveis Independentes / Inputs da máquina)
    'setDynLoad_N',
    'peakDynLoad_N',
    'setStatLoad_N',
    'StatLoad_N',
    'setSpeed_rpm',
    'Speed_rpm',
    
    # 4. Estado do Sistema (Variáveis de degradação lenta)
    'RoomTemp_C',
    'TempT1_C',
    'TempT2_C',
    
    # 5. Sinais de Vibração (Variáveis de alta frequência / final da tabela)
    'AccelA_g',
    'AccelC_g'
]

#df_final = df_final[col_order]

In [ ]:
# --- Exportando dados processados para a pasta destino ---

match file_format:
    case 'csv':
        df_final.to_csv(output_data, index=True)
    
    case "mat":
        scipy.io.savemat(
            output_data,
            {
                **{col: df_final[col].to_numpy()
                for col in df_final.columns},
                "index": df_final.index.to_numpy()
            }
        )
    
    case "parquet":
        df_final.to_parquet(output_data, index=True)
        
    case _:
        raise ValueError(f"Formato '{file_format}' não suportado.")

file_size_mb = output_data.stat().st_size / (1024 * 1024)
print(f"Dataset salvo em: {output_data}")
print(f"{len(df_final) / 1000}k linhas a uma frequência de amostragem efetiva de {128000 / fator_ds} Hz")
print(f"Tamanho do arquivo: {file_size_mb:.2f} MB")


<font size="6">  
    Plots dos dados experimentais 
</font> 

In [ ]:
import matplotlib.pyplot as plt

plot_cols = [
    'peakDynLoad_N',
    'StatLoad_N',
    'Speed_rpm',
    
    'RoomTemp_C',
    'TempT1_C',
    'TempT2_C',
    
    'AccelA_g',
    'AccelC_g'
]

for col in plot_cols:
    plt.figure(figsize=(10,1.5))
    plt.plot(df_final['Rul_s'].max() - df_final['Rul_s'], df_final[col])
    plt.title(col)
    plt.show()

In [ ]:
# Plot de tempos entre medições

from matplotlib.ticker import MaxNLocator

df = df_meta.copy()

# Colunas de delta tempo
df['delta_min'] = df['Time'].diff().dt.total_seconds() / 60
df['delta_mmss'] = (
    df['Time'].diff()
    .dt.total_seconds()
    .fillna(0)
    .astype(int)
    .apply(lambda x: f'{x//60:02d}:{x%60:02d}')
)

# Plot
plt.figure()
plt.plot(df.index, df['delta_min'])

# Force integer ticks on Y axis
plt.gca().yaxis.set_major_locator(MaxNLocator(integer=True))

# Label values greater than 1 minute
for idx, value, label in zip(df.index, df['delta_min'], df['delta_mmss']):
    if value > 1:
        plt.text(idx, value, label, fontsize=8)

plt.xlabel('Measurement')
plt.ylabel('Time Gap (min)')
plt.title('Time Gap Between Consecutive Measurements')
plt.show()

In [ ]:
df = df_final.copy()

# Tempo de experimento em horas, calculado a partir do RUL em segundos
df['exp_time_m'] = (df['Rul_s'].max() - df['Rul_s']) / 60
df['exp_time_h'] = df['exp_time_m'] / 60

# Use o valor absoluto dos sinais de aceleração para ambos os sensores
df['absAccelA_g'] = df['AccelA_g'].abs()
df['absAccelC_g'] = df['AccelC_g'].abs()


plt.figure()

# Linhas de vibração para os dois sensores
plt.plot(df['exp_time_h'], df['absAccelA_g'], label='Sensor A', color='blue')
plt.plot(df['exp_time_h'], df['absAccelC_g'], label='Sensor C', color='orange')

# Intervalo de vibração crítica de falha
plt.axhline(y=6, color='r', linestyle='--', label='Critério de falha por vibração (6-10 g)')
plt.axhline(y=10, color='r', linestyle='--')

# Títulos e rótulos
plt.xlabel('Tempo de Experimento (h)')
plt.ylabel('Vibração Absoluta (g)')
plt.title('Evolução da Vibração Absoluta durante o Experimento')

# Legenda
plt.legend(loc='upper left')

plt.show()

In [ ]:
df = df_final.copy()

# Tempo de experimento em horas, calculado a partir do RUL em segundos
df['exp_time_m'] = (df['Rul_s'].max() - df['Rul_s']) / 60
df['exp_time_h'] = df['exp_time_m'] / 60


plt.figure()

# Linhas de temperatura
plt.plot(df['exp_time_h'], df['TempT1_C'], label='Temperatura Sensor T1', color='blue')
plt.plot(df['exp_time_h'], df['TempT2_C'], label='Temperatura Sensor T2', color='orange')

# Temperatura crítica de falha
plt.axhline(y=110, color='r', linestyle='--', label='Critério de falha: 110°C')

# Títulos e rótulos
plt.xlabel('Tempo de Experimento (h)')
plt.ylabel('Temperatura Absoluta Média (°C)')
plt.title('Evolução da Temperatura durante o Experimento')

# Legenda
plt.legend(loc='lower right')

plt.show()

In [ ]:
df = df_final.copy()

# Tempo de experimento em horas, calculado a partir do RUL em segundos
df['exp_time_m'] = (df['Rul_s'].max() - df['Rul_s']) / 60
df['exp_time_h'] = df['exp_time_m'] / 60

plt.figure(figsize=(10, 1))

# Linhas de velocidade: setada (alvo) vs medida (real)
plt.plot(df['exp_time_h'], df['setSpeed_rpm'], label='Vel. setada', color='red')
plt.plot(df['exp_time_h'], df['Speed_rpm'], label='Vel. medida', color='green')

# Títulos e rótulos
plt.xlabel('Tempo de Experimento (h)')
plt.ylabel('RPM')
plt.title('Evolução da Velocidade de Rotação durante o Experimento')

# Limitando o eixo Y para melhor visualização
#plt.ylim(2200, 3700)

plt.legend()

plt.show()